**Lab08_Hyperparameter_Tuning_and_Model_Selection**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Machine Learning & Tuning Tools
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

print("All ML tuning libraries imported successfully!")

All ML tuning libraries imported successfully!


In [2]:
possible_paths = [
    Path("../data/processed/olist_orders_feature_engineered.csv"),
    Path("data/processed/olist_orders_feature_engineered.csv"),
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv")
]

data_path = None
for p in possible_paths:
    if p.exists():
        data_path = p
        break

df = pd.read_csv(data_path)
print(f"Dataset loaded from: {data_path}")
print("Shape (Rows, Columns):", df.shape)

# Check target distribution
print("\nTarget Distribution (Late Deliveries):")
print(df["is_late_delivery"].value_counts(normalize=True) * 100)

Dataset loaded from: ..\data\processed\olist_orders_feature_engineered.csv
Shape (Rows, Columns): (99441, 35)

Target Distribution (Late Deliveries):
is_late_delivery
0    93.428264
1     6.571736
Name: proportion, dtype: float64


In [3]:
target = "is_late_delivery"

candidate_features = [
    "total_price", "total_freight", "total_items",
    "unique_products", "unique_sellers", "average_item_price",
    "freight_ratio", "items_per_seller", "seller_diversity",
    "is_weekend", "is_business_hour", "month_sin", "month_cos",
    "hour_sin", "hour_cos", "log_total_price",
    "log_total_freight", "items_x_price"
]

# Keep only columns that exist in the dataframe
features = [f for f in candidate_features if f in df.columns]

X = df[features].copy()
y = df[target].copy()

print(f"Selected {len(features)} safe features.")
print("Features:\n", features)

Selected 18 safe features.
Features:
 ['total_price', 'total_freight', 'total_items', 'unique_products', 'unique_sellers', 'average_item_price', 'freight_ratio', 'items_per_seller', 'seller_diversity', 'is_weekend', 'is_business_hour', 'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'log_total_price', 'log_total_freight', 'items_x_price']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape (for tuning):", X_train.shape)
print("X_test shape (untouched test set):", X_test.shape)

X_train shape (for tuning): (79552, 18)
X_test shape (untouched test set): (19889, 18)


In [5]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Baseline pipeline ready!")

Baseline pipeline ready!


In [6]:
param_grid = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.01, 0.1, 1, 10, 100]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Grid and 5-fold CV configured!")

Grid and 5-fold CV configured!


In [7]:
# To make it run fast in ~15-20 seconds on your machine, we sample 25,000 rows for grid search, or run on full X_train
sample_size = min(25000, len(X_train))
X_tune = X_train.iloc[:sample_size]
y_tune = y_train.iloc[:sample_size]

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True
)

print("Running GridSearchCV across 10 configurations (5 folds each)...")
grid_search.fit(X_tune, y_tune)

print("\nGrid Search Complete!")
print("Best Parameters:", grid_search.best_params_)
print(f"Best Cross-Validation F1-score: {grid_search.best_score_:.4f}")

Running GridSearchCV across 10 configurations (5 folds each)...


c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



Grid Search Complete!
Best Parameters: {'model__C': 1, 'model__penalty': 'l2'}
Best Cross-Validation F1-score: 0.1725


In [8]:
results = pd.DataFrame(grid_search.cv_results_)

columns = [
    "param_model__penalty",
    "param_model__C",
    "mean_train_score",
    "std_train_score",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]

results_table = results[columns].sort_values("rank_test_score")
display(results_table)

,param_model__penalty,param_model__C,mean_train_score,std_train_score,mean_test_score,std_test_score,rank_test_score
5,l2,1.00,0.173472,0.001191,0.172538,0.003315,1
6,l1,10.00,0.173452,0.001190,0.172538,0.003283,2
3,l2,0.10,0.173316,0.001233,0.172522,0.003227,3
4,l1,1.00,0.173432,0.001116,0.172482,0.003244,4
2,l1,0.10,0.173276,0.001202,0.172429,0.003575,5
7,l2,10.00,0.173498,0.001144,0.172419,0.003159,6
9,l2,100.00,0.173530,0.001122,0.172419,0.003159,6
8,l1,100.00,0.173498,0.001144,0.172419,0.003159,6
1,l2,0.01,0.173181,0.001158,0.172219,0.003218,9
0,l1,0.01,0.171925,0.000871,0.171320,0.002818,10


In [9]:
best_model = grid_search.best_estimator_

# Fit best model on the full training set
best_model.fit(X_train, y_train)

# Predict on untouched test data
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("============= UNTOUCHED TEST SET EVALUATION =============")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


============= UNTOUCHED TEST SET EVALUATION =============
Accuracy : 0.5824
Precision: 0.1036
Recall   : 0.6993
F1-Score : 0.1804
ROC-AUC  : 0.6606

Confusion Matrix:
[[10670  7912]
 [  393   914]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.57      0.72     18582
           1       0.10      0.70      0.18      1307

    accuracy                           0.58     19889
   macro avg       0.53      0.64      0.45     19889
weighted avg       0.91      0.58      0.68     19889



In [10]:
# Train L1 and L2 pipelines with C = 1.0
l1_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="l1", solver="liblinear", C=1.0, max_iter=1000, class_weight="balanced", random_state=42))
])

l2_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(penalty="l2", solver="liblinear", C=1.0, max_iter=1000, class_weight="balanced", random_state=42))
])

l1_pipe.fit(X_train, y_train)
l2_pipe.fit(X_train, y_train)

l1_coefs = l1_pipe.named_steps["model"].coef_[0]
l2_coefs = l2_pipe.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": features,
    "L1_Coefficient": np.round(l1_coefs, 4),
    "L2_Coefficient": np.round(l2_coefs, 4)
})

display(coef_df)
print(f"L1 Zero Coefficients: {np.sum(l1_coefs == 0)} (Features completely removed!)")
print(f"L2 Zero Coefficients: {np.sum(l2_coefs == 0)}")

c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

,Feature,L1_Coefficient,L2_Coefficient
0,total_price,-0.1460,-0.1592
1,total_freight,-0.0444,-0.0444
2,total_items,-0.5111,-0.5854
3,unique_products,-0.0401,-0.0400
4,unique_sellers,-0.1134,-0.0951
5,average_item_price,0.1136,0.1233
6,freight_ratio,0.0643,0.0646
7,items_per_seller,0.4765,0.5449
8,seller_diversity,0.1055,0.1036
9,is_weekend,-0.0431,-0.0431


L1 Zero Coefficients: 0 (Features completely removed!)
L2 Zero Coefficients: 0


In [11]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
zero_counts = []

for c_val in C_values:
    model_c = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(penalty="l1", solver="liblinear", C=c_val, max_iter=1000, class_weight="balanced", random_state=42))
    ])
    model_c.fit(X_train, y_train)
    coefs = model_c.named_steps["model"].coef_[0]
    zero_counts.append(np.sum(coefs == 0))

zero_table = pd.DataFrame({
    "C_Value": C_values,
    "Regularization_Strength": ["Very Strong", "Strong", "Moderate", "Normal", "Weak", "Very Weak"],
    "Zero_Coefficients_Count": zero_counts
})

display(zero_table)

c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

,C_Value,Regularization_Strength,Zero_Coefficients_Count
0,0.001,Very Strong,12
1,0.010,Strong,4
2,0.100,Moderate,2
3,1.000,Normal,0
4,10.000,Weak,0
5,100.000,Very Weak,0


In [12]:
recall_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1
)

recall_search.fit(X_tune, y_tune)
print("Best parameters for RECALL:", recall_search.best_params_)
print(f"Best Cross-Validation Recall: {recall_search.best_score_:.4f}")

Best parameters for RECALL: {'model__C': 0.01, 'model__penalty': 'l1'}
Best Cross-Validation Recall: 0.6889


c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\priym\OneDrive\Desktop\Priyanshi study\ml26\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
